# Build cache E12 (lưới 136×136×40) — notebook CHỈ để build

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Notebook này **không train gì**. Nó chỉ sinh cache rồi để lại trong
`/kaggle/working/cache_e12` cho bạn lưu thành Kaggle Dataset.

## Vì sao tách khỏi notebook train

Ba lý do, đều là chuyện đã xảy ra chứ không phải phòng xa:

1. **Build ~45 phút ăn vào ngân sách 12h của session train.** Gộp chung thì mỗi
   session chỉ còn ~11h cho train, tức mất gần một fold.
2. **Session chết là mất cache.** Build lại từ đầu ở session sau, mỗi lần 45 phút.
3. **Fold 4–5 chạy ở session khác** (và có thể ở tài khoản Kaggle khác). Cache phải
   là một Dataset mount được, không phải file nằm trong output của một session cụ thể.

Chạy notebook này **một lần**, lưu output thành Dataset, rồi mọi session train sau
chỉ mount nó vào.

## Cache này khác cache E4 ở chỗ nào

| | E4 | **E12** |
|---|---|---|
| lưới lưu trong `.npz` | 112×112×32 | **136×136×40** |
| `crop_margin_voxels` | không có | **[12, 12, 4]** |
| `spacing` suy từ | 112×112×32 | 112×112×32 (**y hệt**) |
| dung lượng | 3,2 GB | ~5,9 GB |

`spacing` không đổi là điều quan trọng nhất: độ phân giải vật lý y hệt E4, phần thêm
ra là **mô thật ở rìa** chứ không phải cùng một khối bị kéo giãn. Hệ quả: cắt giữa
cache E12 cho ra đúng khối mà cache E4 tạo ra, nên val của hai bên so trực tiếp được.

## Cần mount

**Dữ liệu gốc LLD-MMRI** (thư mục chứa `lld/`). Không cần cache cũ, không cần
checkpoint.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

PREPROCESS_NAME = "preprocess_e12.yaml"
CACHE_DIR = Path("/kaggle/working/cache_e12")

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

# build_cache cần SimpleITK (đọc + resample NIfTI). Không cần torch/monai ở đây —
# notebook này không dựng model.
try:
    import SimpleITK  # noqa: F401
    print("SimpleITK: đã có")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "SimpleITK==2.4.0"], check=True)
    print("SimpleITK: vừa cài")

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

PRE = load_yaml(REPO / "configs" / PREPROCESS_NAME)
INNER = tuple(PRE["target_size"])
MARGIN = tuple(PRE["crop_margin_voxels"])
GRID = tuple(s + 2 * m for s, m in zip(INNER, MARGIN))

print(f"\nconfig:     {PREPROCESS_NAME}")
print(f"lưới cache: {GRID}   (target_size {INNER} + lề {MARGIN} mỗi bên)")
print(f"ghi vào:    {CACHE_DIR}")

## 1. Dữ liệu gốc

`resolve_data_root` trả về `config['data_root']` **mà không xác minh** khi mọi cách dò
đều trượt (`src/utils/io.py`). Trên Kaggle nó sẽ là một đường dẫn tương đối không tồn
tại, và job 45 phút sẽ chết giữa chừng. Cell này xác minh trước.

In [ ]:
from src.utils.io import resolve_data_root

cfg_data = load_yaml(REPO / "configs" / "data.yaml")
try:
    data_root = resolve_data_root(cfg_data)
except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo
    data_root, exc_msg = None, str(exc)
else:
    exc_msg = None

ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
if ann is None or not ann.exists():
    raise RuntimeError(
        f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
        f"  resolve_data_root -> {data_root}"
        + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
        + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
        f"  Ứng viên khai trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
    )
print("data root:", data_root, "✓")
print("annotation:", ann.name, "✓")

## 2. Build (~45 phút)

Lớn hơn cache E4 1,84 lần nên lâu hơn cùng tỉ lệ.

**Resume được**: bệnh nhân đã có file `.npz` thì bỏ qua. Session bị ngắt giữa chừng
thì chạy lại đúng cell này, nó đi tiếp từ chỗ dừng.

In [ ]:
import time

t0 = time.time()
rc = subprocess.run(
    [sys.executable, "-m", "src.preprocess.build_cache", "--config", f"configs/{PREPROCESS_NAME}"],
    cwd=REPO,
).returncode
assert rc == 0, "build cache thất bại — đọc log ở trên"
print(f"\nbuild xong sau {(time.time() - t0) / 60:.0f} phút")

## Cổng nghiệm thu ⚠️

Bốn thứ, mỗi thứ chặn một lỗi khác nhau. Cổng này chạy ở đây chứ không ở notebook
train, vì phát hiện cache hỏng sau khi đã upload 5,9 GB là quá muộn.

In [ ]:
import json as _json

import numpy as np

from src.data.transforms import CenterCrop3D

meta = _json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))
CAN = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": list(INNER),
    "crop_margin_voxels": list(MARGIN),
}
for k, v in CAN.items():
    assert meta.get(k) == v, f"cache SAI: {k} = {meta.get(k)!r}, cần {v!r}"
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz}/498 ca — build chưa xong, chạy lại cell trên"

# Hình dạng mảng THẬT. Đây là chỗ bắt được việc `crop_margin_voxels` bị bỏ qua: nếu
# nó không có tác dụng thì mọi thứ khác vẫn đúng, chỉ mảng là 112x112x32.
mau = sorted(CACHE_DIR.glob("*.npz"))[0]
with np.load(mau) as z:
    shape = tuple(z["image"].shape)
    assert shape == (8, *GRID), f"mảng {shape}, cần {(8, *GRID)} — lề dư không có tác dụng"
    assert tuple(z["crop_margin_voxels"]) == MARGIN
    assert tuple(z["inner_size"]) == INNER
    img = z["image"].astype(np.float32)

# Cắt giữa phải ra đúng kích thước model nhận.
cut = CenterCrop3D(INNER)({"image": img})["image"]
assert tuple(cut.shape) == (8, *INNER), cut.shape

tong_gb = sum(f.stat().st_size for f in CACHE_DIR.glob("*.npz")) / 2**30
print(f"✓ {n_npz} ca · mảng {shape} · cắt giữa -> {tuple(cut.shape)} · {tong_gb:.1f} GB")
print(f"  commit build: {meta.get('git_commit')}")

## 3. Lưu thành Kaggle Dataset

⚠️ **Để Private.** LLD-MMRI dùng license CC BY-NC-ND, không được phát tán bản phái sinh.

Hai cách, chọn một:

**A. Save Version của notebook** (dễ nhất). Bấm *Save Version* → *Save & Run All*.
Xong thì output notebook này mount được vào notebook khác qua *Add Data → Your Work →
Notebook Output*. Không cần chạy cell dưới.

**B. Tạo Dataset riêng** bằng CLI, nếu muốn một slug ổn định để mount ở nhiều tài
khoản. Chạy cell dưới rồi chạy lệnh nó in ra.

Dù chọn cách nào, **ghi slug và version vào WORKLOG** — không có dòng đó thì vài tuần
sau không ai biết checkpoint được train bằng cache nào.

In [ ]:
SLUG = "lldmmri-cache-e12"
USER = "marcohoang"          # đổi nếu upload bằng tài khoản khác

(CACHE_DIR / "dataset-metadata.json").write_text(
    _json.dumps({
        "title": "LLD-MMRI cache E12 (136x136x40, le du 12/12/4, per-phase align)",
        "id": f"{USER}/{SLUG}",
        "licenses": [{"name": "other"}],
    }),
    encoding="utf-8",
)

print("Lần đầu:")
print(f"  !kaggle datasets create -p {CACHE_DIR} --dir-mode zip")
print("Các lần sau:")
print(f"  !kaggle datasets version -p {CACHE_DIR} -m 'rebuild' --dir-mode zip")
print("""
Sau khi có Dataset: mở notebooks/14_e12_randomcrop.ipynb, mount cache này vào, và
KHÔNG cần mount dữ liệu gốc nữa. Notebook 14 tự nhận diện cache bằng nội dung
cache_meta.json chứ không bằng tên, nên đặt slug gì cũng được.
""")